# Paper-Ready Metric Plots

PSNR / SSIM vs transmission budget. Error bars = 95% CI of the mean: `mean ± 1.96 × (σ/√n)`.

**Run from `dlapisgs-utility/`:**
```bash
jupyter nbconvert --to notebook --execute --inplace \
    --ExecutePreprocessor.timeout=120 \
    plotting/paper_plot_metrics.ipynb
```

In [1]:
import sys
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use("Agg")

In [2]:
# ------------------------------- setting start ------------------------------ #
color_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
errorbar_color = "#3A3A3A"

# font
csfont = {'family': 'serif', 'serif': ['Times New Roman', 'Times'], 'size': 23}

# errorbar plot size
err_lw       = 1.5
err_capsize  = 4
err_capthick = 1.5

# figure size
figsize = (6.4, 4.8)

# set theme first, then rc — so seaborn doesn't clobber the font size
sns.set_theme(style="ticks", font="Times New Roman")
plt.rc('text', usetex=True)
plt.rc('font', **csfont)
plt.rcParams['text.latex.preamble'] = r'\usepackage{mathptmx}'
# -------------------------------- setting end ------------------------------- #

In [3]:
# ── I/O ──────────────────────────────────────────────────────────────────────
SUMMARY_CSV = "plotting/paper/exp1_weights/source_summary_exp1.csv"
OUT_DIR     = "plotting/paper/exp1_weights/"
GROUP_BY    = "weight_mode"
EXCLUDE_KEYS = ["random"]

BUDGET_PCTS = [10, 25, 40, 55, 70, 85, 99, 100]

color_palette = ["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b","#e377c2","#7f7f7f","#bcbd22","#17becf"]

KEY_CONFIG = {
    "screen_area":      {"label": "Screen Area",                   "marker": "o", "color": color_palette[4]},
    "volume_over_d2":   {"label": r"Volume/$d^2$ (view-dep.)",        "marker": "D", "color": color_palette[3]},
    "volume":           {"label": "Volume (view-indep.)",          "marker": "^", "color": color_palette[2]},
    "random":           {"label": "Random (control)",              "marker": "x", "color": color_palette[7]},
    "vd_lod":           {"label": "Heuristic (baseline)",             "marker": "s", "color": color_palette[0]},
    "vd_lod_w":         {"label": "Heuristic (ours)",                      "marker": "^", "color": color_palette[1]},
    "ml":               {"label": "Learned (ours)",                     "marker": "P", "color": color_palette[2]},
    "oracle_loo":       {"label": "Proxy Oracle",       "marker": "*", "color": color_palette[3]},
    "prog_cull":        {"label": "GS (culled)",    "marker": "o", "color": color_palette[0]},
    "prog_no_cull":     {"label": "GS (not culled)",      "marker": "^", "color": color_palette[2]},
    "tile_strict":      {"label": "Tiles",                   "marker": "s", "color": color_palette[6]},
    "tiled_ml":         {"label": "Tiles Learned",        "marker": "P", "color": color_palette[1]},
    "tiled_oracle":     {"label": "Tiles Oracle",      "marker": "*", "color": color_palette[3]},
}

KEY_ORDER = {
    "weight_mode": ["screen_area", "volume_over_d2", "volume", "random"],
    "scheme":      ["vd_lod", "vd_lod_w", "ml", "oracle_loo"],
    "condition":   ["prog_cull", "prog_no_cull", "tiled_ml", "tiled_oracle"],
}

DPI = 300

In [4]:
import os, sys

# cd to dlapisgs-utility/ regardless of where nbconvert was invoked
_cwd = Path(os.getcwd())
_root = None
_candidates = [_cwd] + list(_cwd.parents) + [Path(p) for p in sys.path]
for _candidate in _candidates:
    if (_candidate / "utility_calculation.py").exists():
        _root = _candidate
        break
if _root is None:
    try:
        _root = Path(__file__).resolve().parent.parent
    except NameError:
        _root = _cwd
os.chdir(_root)
print(f"cwd: {Path.cwd()}")

cwd: /mnt/data1/samk/gs-quic/cs5262_tile_quic/dlapisgs-utility


In [5]:
# shared DATA pipeline: single source of truth with experiments/plot_metrics.py
# (rendering is paper-specific and lives in this notebook, like plot_metric below)
sys.path.insert(0, str(Path.cwd()))
from experiments.plot_metrics import (
    _apply_budget_labels, _aggregate, _resolve_order_and_labels,
    _data_ylim, _bk_sort, PSNR_SATURATION_DB,
)

In [6]:
summary_csv = Path(SUMMARY_CSV)
if not summary_csv.exists():
    raise FileNotFoundError(f"{summary_csv}  (cwd={Path.cwd()})")

df = pd.read_csv(summary_csv)
df = df[~df[GROUP_BY].isin(EXCLUDE_KEYS)].copy()

# convert to list-of-dicts (plot_metrics format) and attach per-scene budget labels
rows = df.to_dict("records")
rows = _apply_budget_labels(rows, BUDGET_PCTS)

print(f"Loaded {len(rows)} rows | groups: {sorted(set(r[GROUP_BY] for r in rows))} | budgets: {sorted(set(r['_budget_key'] for r in rows), key=_bk_sort)}")

Loaded 36000 rows | groups: ['screen_area', 'volume', 'volume_over_d2'] | budgets: ['10%', '25%', '40%', '55%', '70%', '85%', '99%', '100%']


In [7]:
agg    = _aggregate(rows, GROUP_BY)
pm_order, pm_labels = _resolve_order_and_labels(agg, GROUP_BY)

# paper ordering: prefer KEY_ORDER override, fall back to plot_metrics order
preferred = KEY_ORDER.get(GROUP_BY, pm_order)
order = [k for k in preferred if k in agg] + [k for k in pm_order if k not in preferred and k in agg]

# labels: KEY_CONFIG overrides plot_metrics defaults
labels = {k: KEY_CONFIG[k]["label"] if k in KEY_CONFIG else pm_labels.get(k, k) for k in order}

# per-scene aggregates for grid plot
from collections import defaultdict as _dd
_by_scene = _dd(list)
for r in rows:
    _by_scene[r["scene"]].append(r)
SCENE_ORDER = ["bicycle", "garden", "stump", "chair", "drums", "ficus", "hotdog", "materials", "mic", "ship"]
scenes_agg = {s: _aggregate(_by_scene[s], GROUP_BY) for s in SCENE_ORDER if s in _by_scene}

sample_key = order[0]
sample_bk  = sorted(agg[sample_key], key=_bk_sort)[0]
print(f"agg sample  {GROUP_BY}={sample_key!r}  budget={sample_bk}:")
print(agg[sample_key][sample_bk])
print(f"scenes: {list(scenes_agg.keys())}")

agg sample  weight_mode='screen_area'  budget=10%:
{'psnr_mean': 24.095467488606772, 'psnr_ci95': 2.788901183132701, 'ssim_mean': 0.8432109935358166, 'ssim_ci95': 0.028837487077081368, 'ngs_mean': 187667.6, 'ngs_ci95': 160051.17141777815, 'n': 10, 'n_cameras': 1500}
scenes: ['bicycle', 'garden', 'stump', 'chair', 'drums', 'ficus', 'hotdog', 'materials', 'mic', 'ship']


In [8]:
def plot_metric(agg, order, metric, ylabel, out_stem, out_dir):
    fallback_markers = ["s", "^", "D", "o", "v", "P", "X"]

    fig, ax = plt.subplots(figsize=figsize)
    all_means = []

    for i, key in enumerate(order):
        if key not in agg:
            continue
        cfg    = KEY_CONFIG.get(key, {})
        label  = cfg.get("label",  key)
        marker = cfg.get("marker", fallback_markers[i % len(fallback_markers)])
        color  = cfg.get("color",  color_palette[i % len(color_palette)])
        ls     = cfg.get("ls", "-")

        bks = sorted(agg[key], key=_bk_sort)
        xs  = [_bk_sort(bk) for bk in bks]
        ys  = [agg[key][bk][f"{metric}_mean"] for bk in bks]
        es  = [agg[key][bk][f"{metric}_ci95"]  for bk in bks]
        all_means.extend(ys)

        ax.errorbar(xs, ys, yerr=es,
                    marker=marker, color=color, linestyle=ls, linewidth=2.5, markersize=8,
                    capsize=err_capsize, elinewidth=err_lw, capthick=err_capthick,
                    label=label, zorder=2)

    ax.set_xlabel(r"Budget (\% of scene)", fontsize=20)
    ax.set_ylabel(ylabel, fontsize=20)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}\\%"))
    ax.set_ylim(*_data_ylim(all_means, metric))
    if metric == "psnr":
        ax.axhline(PSNR_SATURATION_DB, color="gray", linestyle=":", linewidth=1.5, zorder=0)
        ax.text(ax.get_xlim()[1], PSNR_SATURATION_DB, r" saturation ($\geq$60 dB)",
                fontsize=13, color="gray", va="bottom", ha="right")
    ax.legend(loc="best", framealpha=0.9, fontsize=18)
    ax.tick_params(labelsize=18)

    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.tick_params(direction="out", which="both", top=False, right=False)

    fig.set_constrained_layout(True)
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / f"{out_stem}.png", dpi=DPI, bbox_inches="tight")
    fig.savefig(out_dir / f"{out_stem}.eps", format="eps", bbox_inches="tight")
    print(f"Wrote {out_dir}/{out_stem}.{{png,eps}}")
    plt.close(fig)

In [9]:
def plot_bar(agg, order, metric, ylabel, out_path):
    """Cross-scene grouped bars, paper style (tab10 KEY_CONFIG colors, no title/grid)."""
    bks    = sorted({b for k in agg for b in agg[k]}, key=_bk_sort)
    groups = [k for k in order if k in agg]
    n_b, n_g = len(bks), len(groups)
    width  = 0.8 / max(n_g, 1)
    x      = np.arange(n_b)

    fig, ax = plt.subplots(figsize=(max(8.0, n_b * 1.3), 4.8))
    all_means = []
    for i, key in enumerate(groups):
        cfg = KEY_CONFIG.get(key, {})
        y   = [agg[key].get(b, {}).get(f"{metric}_mean", 0.0) for b in bks]
        err = [agg[key].get(b, {}).get(f"{metric}_ci95",  0.0) for b in bks]
        all_means.extend(v for b, v in zip(bks, y) if b in agg[key])
        offset = (i - n_g / 2 + 0.5) * width
        ax.bar(x + offset, y, width * 0.92, yerr=err, capsize=err_capsize,
               color=cfg.get("color", color_palette[i % len(color_palette)]),
               label=cfg.get("label", key),
               error_kw={"elinewidth": err_lw, "capthick": err_capthick})

    ax.set_ylim(*_data_ylim(all_means, metric))
    ax.set_xticks(x)
    ax.set_xticklabels([b.replace("%", r"\%") for b in bks])
    ax.set_xlabel(r"Budget (\% of scene)", fontsize=20)
    ax.set_ylabel(ylabel, fontsize=20)
    ax.tick_params(labelsize=18)
    ax.legend(fontsize=16, framealpha=0.9)
    ax.set_axisbelow(True)
    if metric == "psnr":
        ax.axhline(PSNR_SATURATION_DB, color="gray", linestyle=":", linewidth=1.5, zorder=0)
        ax.text(n_b - 0.5, PSNR_SATURATION_DB, r"saturation ($\geq$60 dB) ",
                fontsize=13, color="gray", va="bottom", ha="right")

    fig.set_constrained_layout(True)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".eps"), format="eps", bbox_inches="tight")
    print(f"Wrote {out_path}")
    plt.close(fig)

In [10]:
def plot_grid(scenes_agg, order, metric, ylabel, out_path, ncols=5):
    """Per-scene subplot grid, paper style (tab10 colors, shared top legend, no clutter)."""
    scenes = list(scenes_agg.keys())
    n      = len(scenes)
    ncols  = min(ncols, n)
    nrows  = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.4, nrows * 3.0),
                             squeeze=False, sharey=True, constrained_layout=True)
    axes_flat = axes.flatten()
    for ax in axes_flat[n:]:
        ax.set_visible(False)

    ylim = _data_ylim(
        [cell[f"{metric}_mean"]
         for a in scenes_agg.values()
         for budgets in a.values()
         for cell in budgets.values()],
        metric,
    )

    for idx, scene in enumerate(scenes):
        ax = axes_flat[idx]
        a  = scenes_agg[scene]
        bks = next((sorted(a[k], key=_bk_sort) for k in order if k in a), None)
        if bks is None:
            continue
        xs = [_bk_sort(b) for b in bks]
        for key in (k for k in order if k in a):
            cfg = KEY_CONFIG.get(key, {})
            y   = [a[key].get(b, {}).get(f"{metric}_mean", float("nan")) for b in bks]
            err = [a[key].get(b, {}).get(f"{metric}_ci95", 0.0) for b in bks]
            ax.errorbar(xs, y, yerr=err,
                        marker=cfg.get("marker", "o"),
                        color=cfg.get("color", color_palette[0]),
                        linestyle=cfg.get("ls", "-"),
                        linewidth=2, markersize=5, capsize=err_capsize,
                        label=cfg.get("label", key))
        ax.set_title(scene, fontsize=14)
        ax.set_ylim(*ylim)
        if metric == "psnr":
            ax.axhline(PSNR_SATURATION_DB, color="gray", linestyle=":", linewidth=1.0, zorder=0)
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}\\%"))
        ax.tick_params(labelsize=12)
        if idx % ncols == 0:
            ax.set_ylabel(ylabel, fontsize=14)

    # one shared legend on top, collected across panels
    seen = {}
    for ax in axes_flat[:n]:
        for h, l in zip(*ax.get_legend_handles_labels()):
            seen.setdefault(l, h)
    fig.legend(list(seen.values()), list(seen.keys()),
               loc="upper center", ncol=len(seen), fontsize=14,
               framealpha=0.9, bbox_to_anchor=(0.5, 1.12))
    fig.supxlabel(r"Budget (\% of scene)", fontsize=16)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".eps"), format="eps", bbox_inches="tight")
    print(f"Wrote {out_path}")
    plt.close(fig)

In [11]:
out_dir  = Path(OUT_DIR)
grid_dir = out_dir / "_grid"

# 1. aggregate line plot (paper main figure)
plot_metric(agg, order, "psnr", r"Quality in PSNR (dB)", "psnr_vs_budget", out_dir)
plot_metric(agg, order, "ssim", r"Quality in SSIM",       "ssim_vs_budget", out_dir)

# 2. per-scene grid
plot_grid(scenes_agg, order, "psnr", r"Quality in PSNR (dB)", grid_dir / "psnr_vs_budget.png", ncols=5)
plot_grid(scenes_agg, order, "ssim", r"Quality in SSIM", grid_dir / "ssim_vs_budget.png", ncols=5)
print("Done.")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp1_weights/psnr_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp1_weights/ssim_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp1_weights/_grid/psnr_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp1_weights/_grid/ssim_vs_budget.png
Done.


## Wall-Time / Selection Latency (Exp5)

Per-frame selection cost by condition. Two-stage aggregation (scene -> group), same
principle as the PSNR/SSIM `_aggregate` above: scenes are the independent statistical
unit, not cameras, so CI is computed over `n_scenes`, not pooled cameras.

Source: `time_selection.py` full sweep, `output/0702/selection_timing/{scene}/{method}/summary.csv`
(150 cams/method). Scenes are discovered dynamically from the sweep directory and
classified synth/real by the same roster used above (`SCENE_ORDER`) — extending the
sweep to more scenes needs no code changes here, just rerun.

In [ ]:
import matplotlib.patches as mpatches

# ── config ───────────────────────────────────────────────────────────────────
TIMING_SWEEP   = Path("output/0702/selection_timing")
TIMING_OUT_DIR = Path("plotting/paper/timings_selection")

# full canonical roster (same 10 scenes as SCENE_ORDER above). Only chair/bicycle
# have a timing sweep run so far -- the rest are skipped at load time until they do.
TIMING_SCENE_GROUP = {
    "chair": "synth", "drums": "synth", "ficus": "synth", "hotdog": "synth",
    "materials": "synth", "mic": "synth", "ship": "synth",
    "bicycle": "real", "garden": "real", "stump": "real",
}
TIMING_GROUP_LABELS = {"synth": "Synthetic", "real": "Real"}

# condition -> method dir name (same for every scene except tiled_ml -- see override).
TIMING_COND_METHOD = {
    "prog_screen_area": "progressive_screen_area",
    "prog_vol_over_d2": "progressive_vol_d2",
    "tiled_vd_lod":     "vd_lod",
    "tiled_heuristic":  "heuristic",
    "tiled_ml":         "ml_lgbm",
    "tiled_oracle":     "oracle_online",
}
# time_selection.py names the ml dir "ml_{model_type}", but chair's lgbm run
# predates that convention and landed as bare "ml".
TIMING_ML_DIR_OVERRIDE = {"chair": "ml"}
# ml_rf excluded: ml_lgbm is faster on both scenes measured so far and is the
# canonical ML scheme since 2026-07-02 (PLAN.md).

TIMING_COND_LABELS = {
    "prog_screen_area": "Prog.\nScreen Area",
    "prog_vol_over_d2": r"Prog.$\ \mathrm{Vol}/d^2$",
    "tiled_vd_lod":     "Tiled\nBaseline",
    "tiled_heuristic":  "Tiled\nHeuristic",
    "tiled_ml":         "Tiled\nML",
    "tiled_oracle":     "Tiled\nOracle",
}
TIMING_COND_ORDER = list(TIMING_COND_METHOD.keys())

# "utility" = tile_weights_s (aggregate per-GS weight -> per-tile W_k) + utility_s
# (evaluate v/d*W_k) -- per time_selection.py these run back-to-back as one
# "produce a per-tile score" step (heuristic: gaussian_weights -> tile_weights ->
# utility). GS Weights stays separate: it's the upstream per-GS signal computation,
# same role as ML Features plays for the ML scheme.
# "render" (oracle's actual LOO renders) is a different operation that happens to
# share the same pipeline slot as "utility" -- kept as its own stage, not merged in.
# colors: reuse color_palette (defined above) by index, not an invented palette.
TIMING_STAGE_DEFS = [
    ("visibility", "Visibility",      color_palette[0]),
    ("gw",         "GS Weights",      color_palette[1]),
    ("ml_feat",    "ML Features",     color_palette[2]),
    ("utility",    "Tile Utility", color_palette[3]),
    ("render",     "LOO Render",      color_palette[4]),
    ("greedy_sel", "Greedy Select",   color_palette[5]),
]

In [13]:
# ── load per-camera timing summaries, one row per (scene, condition) ───────
_timing_rows = []
_timing_missing = []
for scene, group in TIMING_SCENE_GROUP.items():
    for cond, method in TIMING_COND_METHOD.items():
        if cond == "tiled_ml":
            method = TIMING_ML_DIR_OVERRIDE.get(scene, method)
        csv = TIMING_SWEEP / scene / method / "summary.csv"
        if not csv.exists():
            _timing_missing.append(f"{scene}/{cond}")
            continue
        mdf = pd.read_csv(csv).fillna(0.0)
        _timing_rows.append({
            "scene": scene, "group": group, "condition": cond, "n_cameras": len(mdf),
            "total_ms":   mdf["total_s"].mean() * 1000,
            "visibility": mdf["visibility_s"].mean() * 1000,
            "gw":         mdf["gaussian_weights_s"].mean() * 1000,
            "ml_feat":    (mdf["ml_group_a_s"] + mdf["ml_predict_s"]).mean() * 1000,
            "utility":    (mdf["tile_weights_s"] + mdf["utility_s"]).mean() * 1000,
            "render":     (mdf["full_render_s"] + mdf["tile_renders_s"]).mean() * 1000,
            "greedy_sel": mdf["greedy_s"].mean() * 1000,
        })

timing_df = pd.DataFrame(_timing_rows)
print(f"loaded {len(timing_df)} (scene,condition) rows | scenes present: {sorted(timing_df['scene'].unique())}")
if _timing_missing:
    print(f"not yet swept, skipped: {_timing_missing}")
timing_df.head()

loaded 12 (scene,condition) rows | scenes present: ['bicycle', 'chair']
not yet swept, skipped: ['drums/prog_screen_area', 'drums/prog_vol_over_d2', 'drums/tiled_vd_lod', 'drums/tiled_heuristic', 'drums/tiled_ml', 'drums/tiled_oracle', 'ficus/prog_screen_area', 'ficus/prog_vol_over_d2', 'ficus/tiled_vd_lod', 'ficus/tiled_heuristic', 'ficus/tiled_ml', 'ficus/tiled_oracle', 'hotdog/prog_screen_area', 'hotdog/prog_vol_over_d2', 'hotdog/tiled_vd_lod', 'hotdog/tiled_heuristic', 'hotdog/tiled_ml', 'hotdog/tiled_oracle', 'materials/prog_screen_area', 'materials/prog_vol_over_d2', 'materials/tiled_vd_lod', 'materials/tiled_heuristic', 'materials/tiled_ml', 'materials/tiled_oracle', 'mic/prog_screen_area', 'mic/prog_vol_over_d2', 'mic/tiled_vd_lod', 'mic/tiled_heuristic', 'mic/tiled_ml', 'mic/tiled_oracle', 'ship/prog_screen_area', 'ship/prog_vol_over_d2', 'ship/tiled_vd_lod', 'ship/tiled_heuristic', 'ship/tiled_ml', 'ship/tiled_oracle', 'garden/prog_screen_area', 'garden/prog_vol_over_d2', 'ga

,scene,group,condition,n_cameras,total_ms,visibility,gw,ml_feat,utility,render,greedy_sel
0,chair,synth,prog_screen_area,150,7.691054,0.567169,3.928346,0.000000,0.000000,0.0,3.195538
1,chair,synth,prog_vol_over_d2,150,2.792876,0.565520,1.543977,0.000000,0.000000,0.0,0.683378
2,chair,synth,tiled_vd_lod,150,1.756918,0.558083,0.000000,0.000000,0.139481,0.0,1.059353
3,chair,synth,tiled_heuristic,150,8.424995,0.567010,3.963316,0.000000,2.945439,0.0,0.949231
4,chair,synth,tiled_ml,150,5.882934,0.588121,0.000000,4.263652,0.000000,0.0,1.031161


In [14]:
# ── aggregate: group mean per condition (+ CI once >1 scene/group exists) ──
def _ci95(vals):
    n = len(vals)
    return 1.96 * float(np.std(vals, ddof=1)) / np.sqrt(n) if n > 1 else 0.0


timing_bar_agg = {"synth": {}, "real": {}}
for group in timing_bar_agg:
    gdf = timing_df[timing_df["group"] == group]
    for cond in TIMING_COND_ORDER:
        vals = gdf.loc[gdf["condition"] == cond, "total_ms"].to_numpy()
        timing_bar_agg[group][cond] = {"mean": float(np.mean(vals)), "ci95": _ci95(vals)}


def timing_stage_group_means(group):
    """Per-condition stage-mean ms, averaged across scenes in `group`."""
    gdf = timing_df[timing_df["group"] == group]
    return {
        cond: {sk: float(gdf.loc[gdf["condition"] == cond, sk].mean()) for sk, _, _ in TIMING_STAGE_DEFS}
        for cond in TIMING_COND_ORDER
    }


print(timing_bar_agg)

{'synth': {'prog_screen_area': {'mean': 7.691053586701507, 'ci95': 0.0}, 'prog_vol_over_d2': {'mean': 2.792876151700766, 'ci95': 0.0}, 'tiled_vd_lod': {'mean': 1.7569176480173583, 'ci95': 0.0}, 'tiled_heuristic': {'mean': 8.424994734426289, 'ci95': 0.0}, 'tiled_ml': {'mean': 5.882934418817313, 'ci95': 0.0}, 'tiled_oracle': {'mean': 2703.983730326096, 'ci95': 0.0}}, 'real': {'prog_screen_area': {'mean': 242.74229409794012, 'ci95': 0.0}, 'prog_vol_over_d2': {'mean': 107.51735568046566, 'ci95': 0.0}, 'tiled_vd_lod': {'mean': 53.653591498732524, 'ci95': 0.0}, 'tiled_heuristic': {'mean': 284.6978173777461, 'ci95': 0.0}, 'tiled_ml': {'mean': 70.79830611745513, 'ci95': 0.0}, 'tiled_oracle': {'mean': 2443.982278952996, 'ci95': 0.0}}}


In [15]:
def plot_timing_bar(bar_agg, cond_order, cond_labels, group_labels, out_path):
    """Grouped bar, log y-axis (latency spans ~3 orders of magnitude: oracle vs the rest)."""
    groups = list(group_labels.keys())
    conds = [c for c in cond_order if any(c in bar_agg[g] for g in groups)]

    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    n = len(conds)
    x = np.arange(n)
    w = 0.8 / max(len(groups), 1)

    for i, g in enumerate(groups):
        offset = (i - (len(groups) - 1) / 2) * w
        means = [bar_agg[g].get(c, {}).get("mean", np.nan) for c in conds]
        cis   = [bar_agg[g].get(c, {}).get("ci95", 0.0) for c in conds]
        ax.bar(x + offset, means, w * 0.92, yerr=cis,
               error_kw={"elinewidth": err_lw, "capthick": err_capthick, "capsize": err_capsize},
               color=color_palette[i], label=group_labels[g])

    ax.set_yscale("log")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v)}" if v >= 1 else f"{v:.1f}"))
    ax.set_xticks(x)
    ax.set_xticklabels([cond_labels[c] for c in conds], fontsize=13)
    ax.set_ylabel(r"Selection latency (ms/frame)", fontsize=18)
    ax.tick_params(axis="y", labelsize=16, which="major")
    ax.tick_params(axis="y", which="minor", length=3)
    ax.tick_params(axis="x", which="both", top=False, length=0)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.legend(fontsize=14, framealpha=0.9, loc="upper left")

    fig.set_constrained_layout(True)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".eps"), format="eps", bbox_inches="tight")
    print(f"Wrote {out_path}")
    plt.close(fig)


def plot_timing_stages(stage_means_by_cond, cond_order, cond_labels, title, out_path):
    """Per-stage stacked bar with a broken y-axis (oracle ~2500ms dwarfs the rest, <300ms)."""
    conds = [c for c in cond_order if c in stage_means_by_cond]
    totals = [sum(stage_means_by_cond[c].values()) for c in conds]
    non_oracle_max = max(t for c, t in zip(conds, totals) if c != "tiled_oracle")
    break_y = non_oracle_max * 1.35
    top_max = max(totals) * 1.22  # extra headroom so the top value label doesn't clip the spine

    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, figsize=(7.5, 6.0),
        gridspec_kw={"height_ratios": [1, 2.2], "hspace": 0.08},
        constrained_layout=False,
    )

    n = len(conds)
    x = np.arange(n)
    w = 0.55
    for ax in (ax_top, ax_bot):
        bottoms = np.zeros(n)
        for sk, label, color in TIMING_STAGE_DEFS:
            heights = np.array([stage_means_by_cond[c][sk] for c in conds])
            ax.bar(x, heights, w, bottom=bottoms, color=color, label=label)
            bottoms += heights

    ax_bot.set_ylim(0, break_y)
    ax_top.set_ylim(break_y, top_max)

    for i, total in enumerate(totals):
        if total > break_y:
            ax, off = ax_top, (top_max - break_y) * 0.03
        else:
            ax, off = ax_bot, break_y * 0.02
        ax.text(i, total + off, f"{total:.1f}", ha="center", va="bottom", fontsize=11, color="#333333")

    ax_bot.set_xticks(x)
    ax_bot.set_xticklabels([cond_labels[c] for c in conds], fontsize=12)
    ax_top.set_title(title, fontsize=14, pad=8)
    ax_bot.set_ylabel(r"Selection latency (ms/frame)", fontsize=14)
    ax_bot.tick_params(axis="y", labelsize=13)
    ax_top.tick_params(axis="y", labelsize=13)
    ax_bot.tick_params(axis="x", length=0)

    ax_top.spines["bottom"].set_visible(False)
    ax_bot.spines["top"].set_visible(False)
    ax_top.spines["right"].set_visible(False)
    ax_bot.spines["right"].set_visible(False)
    ax_top.tick_params(axis="x", bottom=False, labelbottom=False)

    d = 0.012
    kw = dict(transform=ax_top.transAxes, color="k", clip_on=False, linewidth=1.2)
    ax_top.plot((-d, +d), (-2 * d, +2 * d), **kw)
    kw = dict(transform=ax_bot.transAxes, color="k", clip_on=False, linewidth=1.2)
    ax_bot.plot((-d, +d), (1 - d, 1 + d), **kw)

    # legend goes in ax_top's blank region (only the rightmost/oracle bar has content there)
    handles = [mpatches.Patch(color=c, label=lbl) for _, lbl, c in TIMING_STAGE_DEFS]
    ax_top.legend(handles=handles, loc="upper left", ncol=2, fontsize=10, framealpha=0.9)

    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".eps"), format="eps", bbox_inches="tight")
    print(f"Wrote {out_path}")
    plt.close(fig)

In [16]:
plot_timing_bar(timing_bar_agg, TIMING_COND_ORDER, TIMING_COND_LABELS,
                 TIMING_GROUP_LABELS, TIMING_OUT_DIR / "selection_timing_bar.png")

for group, title in TIMING_GROUP_LABELS.items():
    stage_means = timing_stage_group_means(group)
    plot_timing_stages(stage_means, TIMING_COND_ORDER, TIMING_COND_LABELS, title,
                        TIMING_OUT_DIR / f"selection_timing_stages_{group}.png")

print("Done.")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/timings_selection/selection_timing_bar.png


/tmp/ipykernel_3298210/2288889907.py:97: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/timings_selection/selection_timing_stages_synth.png


/tmp/ipykernel_3298210/2288889907.py:97: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/timings_selection/selection_timing_stages_real.png
Done.
